In [2]:
import os
import pandas as pd
from tqdm.auto import tqdm
tqdm.pandas()

data_full = pd.read_csv("reviews_1estrella_2023_2025.csv")

text_column_name = "texto"
data_full = data_full.dropna(subset=[text_column_name]).copy()
data_full = data_full[data_full[text_column_name].astype(str).str.strip() != ""]
data_full = data_full.reset_index(drop=True)
data_full.index.name = "review_id"

SIZE_OF_SAMPLE_FOR_HANDCODING = 150
data_sample = data_full.sample(n=min(SIZE_OF_SAMPLE_FOR_HANDCODING, len(data_full)), random_state=613).copy()

print(f"Total: {len(data_full)} reseñas | Muestra: {len(data_sample)}")

Total: 748 reseñas | Muestra: 150


In [37]:
from enum import Enum
from typing import List
from pydantic import BaseModel

prompt_base = """
Sos un experto en analizar reseñas de hospitales públicos de Argentina. Vas a leer una reseña negativa (de 1 estrella) e identificar TODOS los motivos de queja presentes en el texto, eligiendo entre estas categorías: {categories}.

PRINCIPIO GENERAL: Sé conservador y específico. Asigná una categoría SOLO cuando el texto la respalda con una señal CONCRETA (una acción, un desenlace, un hecho descripto). Si la queja es vaga, genérica o no se puede determinar con precisión, usá "other". Ante la duda, preferí "other" antes que asignar una categoría específica.

Definiciones:

- "incorrect_diagnoses": el texto describe concretamente un diagnóstico equivocado, errado o no detectado a tiempo (le dijeron que tenía una cosa y era otra, no detectaron algo grave, un error de diagnóstico que luego se corrige en otro lado).

- "delays": el texto hace referencia concreta a demoras molestas o tiempos de espera que, para el paciente, fueron prolongados (horas de espera, turnos que tardan meses, no ser atendido tras esperar mucho, lentitud explícita en la atención).

- "inefficient_medical_treatment": usá esta categoría SOLO si el texto menciona un DESENLACE MÉDICO concreto que muestre que la atención no resolvió el problema de salud: el paciente no mejoró, empeoró, tuvo que ir a otro hospital o clínica para resolverlo, le dieron una medicación equivocada, sufrió una complicación por la atención recibida, o hubo un daño concreto (incluida la muerte). Debe haber un resultado clínico descripto, no solo una queja sobre la calidad. NO uses esta categoría para quejas genéricas sobre "mala atención médica" sin un desenlace concreto.

- "bad_infrastructure": el texto hace referencia concreta a mal equipamiento, mala higiene, escasez de recursos o mal funcionamiento del sistema del hospital (suciedad, falta de insumos, aparatos que no funcionan, falta de camas, instalaciones deterioradas, sistemas administrativos que no funcionan).

- "inappropriate_interpersonal_behavior": usá esta categoría SOLO si el texto describe una ACCIÓN o CONDUCTA concreta de una persona que trabaja en el hospital que constituya mal trato: gritó, insultó, se burló, contestó de mala manera, echó a alguien, se negó a escuchar, ignoró a propósito a quien le hablaba, hizo un gesto despectivo, discriminó. Tiene que haber un VERBO que describa qué hizo esa persona. NO uses esta categoría si el texto solo contiene adjetivos o calificaciones genéricas sin describir la acción ("mala atención", "pésimo trato", "una vergüenza", "denigrante", "no te dan bola"): esas van a "other", o a la categoría concreta correspondiente si el fondo es una demora o un problema médico.

- "other": usá esta categoría cuando NINGUNA de las anteriores se cumpla de forma concreta. Esto incluye quejas vagas o genéricas que expresan malestar sin especificar el motivo ("pésimo servicio", "un desastre", "una vergüenza", "no lo recomiendo", "desorganizados"), y quejas sobre cosas o personas ajenas al funcionamiento médico del hospital.

Reglas IMPORTANTES:
1. Una reseña PUEDE pertenecer a VARIAS categorías al mismo tiempo. Incluí TODAS las que apliquen de forma concreta.
2. Ejemplo: si describe un diagnóstico errado que además derivó en un mal tratamiento, incluí "incorrect_diagnoses" Y "inefficient_medical_treatment".
3. Un comentario genérico ("pésima atención") acompañado de una queja específica (por ejemplo una demora concreta) se clasifica SOLO por la queja específica ("delays"). La parte genérica NO agrega ninguna categoría.
4. Si una reseña dice "mala atención" pero el fondo es que lo hicieron esperar, es "delays". Si el fondo es que no le resolvieron el problema de salud, es "inefficient_medical_treatment". No es "inappropriate_interpersonal_behavior" salvo que describa una situación concreta de maltrato.
5. Usá "other" ÚNICAMENTE cuando ninguna de las otras cinco categorías aplique. NUNCA combines "other" con otra categoría.
6. Basate SOLO en el texto de la reseña.
7. Devolvé al menos una categoría.

Reseña a clasificar:
{tweet_text}
"""

class HospitalReviewOptions(str, Enum):
    incorrect_diagnoses = "incorrect_diagnoses"
    delays = "delays"
    inefficient_medical_treatment = "inefficient_medical_treatment"
    bad_infrastructure = "bad_infrastructure"
    inappropriate_interpersonal_behavior = "inappropriate_interpersonal_behavior"
    other = "other"

class HospitalReviewValidOptions(BaseModel):
    classifications: List[HospitalReviewOptions]

data_sample_prompt_column = data_sample.apply(lambda row: prompt_base.format(
    tweet_text=row[text_column_name],
    categories=", ".join(['"{}"'.format(opt.value) for opt in HospitalReviewOptions])
), axis="columns")
full_data_prompt_column = data_full.apply(lambda row: prompt_base.format(
    tweet_text=row[text_column_name],
    categories=", ".join(['"{}"'.format(opt.value) for opt in HospitalReviewOptions])
), axis="columns")

In [38]:
from dotenv import load_dotenv
load_dotenv()

import os
print("Key encontrada:", bool(os.environ.get("OPENROUTER_API_KEY_LEDE")))

Key encontrada: True


In [39]:
import os
from openrouter import OpenRouter

openrouter_client = OpenRouter(api_key=os.environ.get("OPENROUTER_API_KEY_LEDE"))

MODEL_TO_USE = "gpt-5.4-mini"  
openrouter_model_names = {
    "gpt-5.4-mini": "openai/gpt-5.4-mini",
    "gpt-5.4": "openai/gpt-5.4",
    "gpt-5.4-nano": "openai/gpt-5.4-nano",
    "claude-4.5-haiku": "anthropic/claude-4.5-haiku",
}

def classify(prompt_including_review):
    response = openrouter_client.chat.send(
        model=openrouter_model_names[MODEL_TO_USE],
        messages=[{"role": "user", "content": prompt_including_review}],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "hospital_review_classification",
                "strict": True,
                "schema": {
                    "type": "object",
                    "properties": {
                        "classifications": {
                            "type": "array",
                            "items": {
                                "type": "string",
                                "enum": [opt.value for opt in HospitalReviewOptions]
                            }
                        }
                    },
                    "required": ["classifications"],
                    "additionalProperties": False
                }
            }
        }
    )
    import json
    try:
        parsed = json.loads(response.choices[0].message.content)
        return parsed["classifications"]
    except Exception:
        return []

In [40]:
data_sample["ai_guess"] = data_sample_prompt_column.progress_apply(classify)

# ver algunos resultados
import pandas as pd
with pd.option_context("display.max_colwidth", 400):
    display(data_sample[[text_column_name, "ai_guess"]].head(10))

  0%|          | 0/150 [00:00<?, ?it/s]

,texto,ai_guess
review_id,,
420,"Hospital abandonado, es deprimente cómo están las instalaciones. El destrato de recepcionistas de informes y de las especialidades, indigna.","[bad_infrastructure, inappropriate_interpersonal_behavior]"
292,"Приехали с острой болью, сажали что они не работаю в 7:30 ждите 8 утра и все поливать . Не советую худшая госпиталь в Аргентине самый плохой . Можно было бы поставить - 5",[delays]
537,"Desastre. No tienen guardia de las especialidades por las que derivan otros hospitales. La atención en recepción es pésima. A la mujer que atiende le molesta que le hagan preguntas e ignora a propósito a la gente (irónicamente coloca un cartel a birome que dice ""diríjase con respeto""). Debería conseguir otro trabajo si tanto le fastidia la atención al público, pero supongo que sin capacidades ...","[inappropriate_interpersonal_behavior, other]"
0,"Ayer estuve en el Hospital Álvarez con mi cuñado, que es esquizofrénico. Estaba en crisis, quería suicidarse, y se cayó, fracturándose el brazo izquierdo. Desde las 13 hs hasta las 19 hs nadie nos atendió. Se cayó al piso y ni el camillero ni los médicos se acercaron; pasaban por su lado y se reían. Nadie nos ayudó. Después de seis horas nos atendió un supuesto psiquiatra, que hizo las recetas...","[delays, inappropriate_interpersonal_behavior, other]"
74,"La verdad es un desastre la guardia, tardan 3 horas en atenderte (con suerte), el trato es malísimo, la doctora de guardia una incompetente bárbara, el chico de administración contesta de muy mala manera. La verdad inhumano el trato.","[delays, inappropriate_interpersonal_behavior]"
401,"""Tuve una mala experiencia en el hospital Durán debido a la atención deficiente y el destrato del personal médico y administrativo. Recuerden que el juramento hipocrático implica un compromiso de respeto y cuidado hacia los pacientes. Necesitan mejorar en la atención.""",[inappropriate_interpersonal_behavior]
53,"Una vergüenza en guardia hoy fuimos a ls 8:30AM, para traumatolo, y eran 8 personas para esa espesalidad, nos tuvimos q retirar casi la mayoría, pq eran 15:10PM y no. Fueron atendidos , cada rato preguntaban y decían. Enseguida vienen, UNA VERGÜENZA",[delays]
19,"La persona de admisión se va y deja sin atender el lugar.\nLos enfermeros tienen un super destrato hacia las personas, sin causa. Simplemente se colocan en una posición de superioridad tratando despectivamente a los pacientes, sin brindar la información de forma clara. Fui con mi mamá a la guardia, le ordenaron laboratorio. 4 horas reloj esperando el resultado.","[inappropriate_interpersonal_behavior, delays]"
651,"Si necesitan atencion cardiologica, NO SE ATIENDAN CON EL DOCTOR BLANCO, es lo peor que les puede pasar.\nVayan solo si les gusta que los maltraten y los boludeen.\nExperiencia nefasta con este ""Dr"".",[inappropriate_interpersonal_behavior]


In [41]:
def limpiar_other(lista):
    """Si 'other' aparece junto a otras categorías, la saca (other debe ir sola)."""
    if isinstance(lista, list) and len(lista) > 1 and "other" in lista:
        return [c for c in lista if c != "other"]
    return lista

data_sample["ai_guess"] = data_sample["ai_guess"].apply(limpiar_other)

quedan = data_sample["ai_guess"].apply(lambda l: "other" in l and len(l) > 1).sum()
print(f"Casos con 'other' mal combinado que quedan: {quedan}")   # debe dar 0

Casos con 'other' mal combinado que quedan: 0


In [42]:
export = data_sample.copy()

export["ai_guess"] = export["ai_guess"].apply(lambda l: "; ".join(l) if isinstance(l, list) else "")

export["groundtruth"] = ""

export = export[["texto", "ai_guess", "groundtruth"]]
export.to_csv("muestra_para_codificar.csv", index=False, encoding="utf-8-sig")
print(f"Exportadas {len(export)} reseñas a muestra_para_codificar.csv")

Exportadas 150 reseñas a muestra_para_codificar.csv


In [43]:
print(f"Reseñas a clasificar: {len(data_full)}")

Reseñas a clasificar: 748


In [45]:
data_full["ai_guess"] = full_data_prompt_column.progress_apply(classify)
data_full["ai_guess"] = data_full["ai_guess"].apply(limpiar_other)
print(f"Clasificadas: {len(data_full)} reseñas")

  0%|          | 0/748 [00:00<?, ?it/s]

Clasificadas: 748 reseñas


In [46]:
data_full["categorias"] = data_full["ai_guess"].apply(
    lambda l: "; ".join(l) if isinstance(l, list) else str(l))
data_full.to_csv("reviews_748_clasificadas.csv", index=False, encoding="utf-8-sig")
print(f"Guardado: reviews_748_clasificadas.csv con {len(data_full)} reseñas")
data_full[["texto", "categorias", "hospital", "fecha_iso", "rating"]].head()

Guardado: reviews_748_clasificadas.csv con 748 reseñas


,texto,categorias,hospital,fecha_iso,rating
review_id,,,,,
0,Ayer estuve en el Hospital Álvarez con mi cuña...,delays; inappropriate_interpersonal_behavior,Alvarez,2025-10-29T10:02:19Z,1.0
1,"Desastre total la atención, la traje a mi pare...",delays; inappropriate_interpersonal_behavior,Alvarez,2025-04-12T07:42:46Z,1.0
2,Un desastre. Realmente un desastre. Fuimos con...,inappropriate_interpersonal_behavior,Alvarez,2024-07-11T01:57:06Z,1.0
3,Un desastre de admisión estuvieron haciendo es...,delays; inappropriate_interpersonal_behavior,Alvarez,2025-07-09T09:32:49Z,1.0
4,"es una vergüenza este hospital, hace 3hs que e...",delays; inappropriate_interpersonal_behavior,Alvarez,2025-12-25T20:06:14Z,1.0


In [47]:
#Check everything is categoryzed

In [48]:
vacias = data_full["ai_guess"].apply(lambda l: len(l) == 0 if isinstance(l, list) else True).sum()
print(f"Reseñas sin categoría: {vacias}")

Reseñas sin categoría: 0


In [51]:
solo_delays = data_full[data_full["ai_guess"].apply(lambda l: l == ["delays"])]

print(f"Reseñas que hablan solo de delays: {len(solo_delays)}\n")
solo_delays[["hospital", "fecha_iso", "texto"]].reset_index(drop=True)

Reseñas que hablan solo de delays: 145



,hospital,fecha_iso,texto
0,Alvarez,2024-03-13T09:00:47Z,"Un desastre total , la guardia un examen de la..."
1,Alvarez,2025-11-10T14:32:06Z,Muy mala atención tenía turno para hoy para ha...
2,Alvarez,2025-03-25T16:23:45Z,La guardia nocturna un desastre. Fui por presi...
3,Alvarez,2024-12-07T15:32:31Z,"Hoy es sábado, la señora de admisión me anotó ..."
4,Alvarez,2024-10-13T23:04:08Z,Una vergüenza en guardia hoy fuimos a ls 8:30A...
...,...,...,...
140,Rivadavia,2024-04-26T03:32:16Z,Un desastre estuvimos 5 hs esperando y atendía...
141,Rivadavia,2023-03-01T03:37:49Z,Desastre. Vas por Guardia te presentas con par...
142,Rivadavia,2025-02-22T00:51:01Z,"Pésima atención en la guardia, puedes durar ha..."
143,Rivadavia,2025-03-23T16:56:07Z,Hace una hora y media de espera… dicen que los...


In [53]:
data_full["anio"] = pd.to_datetime(data_full["fecha_iso"]).dt.year

solo_delays_2025 = data_full[
    (data_full["ai_guess"].apply(lambda l: l == ["delays"])) &
    (data_full["anio"] == 2025)
]

print(f"Reseñas de solo-delays en 2025: {len(solo_delays_2025)}\n")

for i, row in solo_delays_2025.iterrows():
    print(f"[{row['hospital']} — {row['fecha_iso'][:10]}]")
    print(row["texto"])
    print("-" * 80)

Reseñas de solo-delays en 2025: 47

[Alvarez — 2025-11-10]
Muy mala atención tenía turno para hoy para hacerme un PAP y colposcopia y estaba horario y no me atendieron un desastre total por dios
--------------------------------------------------------------------------------
[Alvarez — 2025-03-25]
La guardia nocturna un desastre. Fui por presión alta, no había nadie que te atienda, como 100 personas esperando. Esperé dos horas, nunca apareció ni medico ni enfermeros. Un desastre como todo lo que está manejando este gobierno de la ciudad, el cual es de una ineptitud nunca vista en caba.
--------------------------------------------------------------------------------
[Alvarez — 2025-04-03]
Fui con turno a darme la vacuna del covid y resulta que ya no habia numeros, el turno no asegura que me reserven la vacuna?. Entonces para que ir con turno?. No entiendo.
--------------------------------------------------------------------------------
[Alvarez — 2025-01-19]
Mejor no venir muriendo acá,

In [54]:
fragmento = "te podrían hasta velar en el hospital"

encontrada = data_full[data_full["texto"].str.contains(fragmento, case=False, na=False)]

print(f"Reseñas encontradas: {len(encontrada)}\n")
for i, row in encontrada.iterrows():
    print(f"Hospital: {row['hospital']}")
    print(f"Fecha exacta: {row['fecha_iso']}")
    print(f"Fecha relativa: {row['fecha_relativa']}")
    print(f"Rating: {row['rating']}")
    print(f"Autor: {row['autor']}")
    print(f"Categorías IA: {row['ai_guess']}")
    print(f"\nTexto:\n{row['texto']}")

Reseñas encontradas: 1

Hospital: Penna
Fecha exacta: 2025-05-10T09:17:21Z
Fecha relativa: Hace un año
Rating: 1.0
Autor: Fulvio Torres
Categorías IA: ['delays']

Texto:
Llegamos a las 3:20 de la madrugada y recién nos atendieron a las 9:00 de la mañana. La atención de los médicos/suplentes es un asco. En pocas palabras si te estás muriendo, te podrían hasta velar en el hospital por las horas esperadas.


In [56]:
completo = pd.read_csv("reviews_todos.csv")
completo["fecha_dt"] = pd.to_datetime(completo["fecha_iso"])
print(f"Fecha más reciente de todo el dataset: {completo['fecha_dt'].max()}")

Fecha más reciente de todo el dataset: 2026-07-01 01:52:58+00:00
